# How the Portugal housing project was built, step by step

**The question.** How far have house prices in Portugal outrun incomes, compared with six peer countries and the EU,
and who feels it?

**How this notebook is organised.** It follows the project in the order it was built:

| Step | What | Tool |
|---|---|---|
| 1 | Download five Eurostat tables through the statistics API | Python standard library |
| 2 | Load the raw files and check coverage and status flags | pandas |
| 3 | Model the indices in dbt: quarters, real prices, price-to-income | dbt on DuckDB |
| 4 | Answer the question with interactive charts | pandas + Altair |
| 5 | Reconcile pandas with the dbt marts and the published reference numbers | pandas + DuckDB |
| 6 | Publish: web dashboard, Power BI and Qlik | JSON export, PBIP, Qlik script |

Unlike the other projects in the portfolio, the data here are **real**: official Eurostat statistics for Portugal,
Spain, Ireland, Italy, France, Germany, the Netherlands and the EU-27, downloaded on 24 October 2026.

**The charts are interactive**: switch between real and nominal prices, click a country to highlight it or to drive
the linked chart. They are built with [Altair](https://altair-viz.github.io/); open the
[interactive version](https://guilhermerigonato.com/pt-housing-analytics/notebook/) to use them (GitHub shows
notebooks without JavaScript).

**Run it:** `pip install pandas altair duckdb dbt-duckdb`, then from the repository root
`python scripts/get_data.py` and `dbt build --profiles-dir .`, and run all cells (a few seconds).

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import altair as alt

RAW = Path("../data/raw")
pd.set_option("display.float_format", "{:,.1f}".format)
NAMES = {"PT": "Portugal", "ES": "Spain", "IE": "Ireland", "IT": "Italy", "FR": "France", "DE": "Germany",
         "NL": "Netherlands", "EU27_2020": "EU-27"}
ORDER = list(NAMES.values())
BLUE, ORANGE, GREEN, GREY, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#c9c8c3", "#52514e", "#e4e3df"
PALETTE = alt.Scale(domain=ORDER, range=[ORANGE, BLUE, GREEN, "#9a6fd0", "#c2a34a", "#5aa5c9", "#d77fa1", MUTED])

@alt.theme.register("portfolio", enable=True)
def portfolio_theme():
    axis = {"gridColor": GRID, "domainColor": MUTED, "tickColor": MUTED, "labelColor": MUTED,
            "titleColor": MUTED, "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": "normal"}
    return {"config": {"view": {"stroke": None}, "axis": axis, "legend": {"labelColor": MUTED, "titleColor": MUTED},
                       "title": {"anchor": "start", "fontSize": 13, "color": "#0b0b0b"},
                       "font": "system-ui, -apple-system, Segoe UI, sans-serif"}}

## Step 1. Download the data

[`scripts/get_data.py`](../scripts/get_data.py) calls the Eurostat dissemination API, which answers in JSON-stat:
a flat array of values plus, for each dimension, the list of codes and their positions. The script unrolls that into
a long CSV (one column per dimension, then period, value and status flag) using only the standard library.

| Table | What | Frequency |
|---|---|---|
| `prc_hpi_q` | House price index, 2015 = 100, all dwellings, new and existing | Quarterly |
| `prc_hicp_minr` | Consumer prices (HICP), all items and actual rentals (ECOICOP ver. 2) | Monthly |
| `ilc_lvho07a` | Housing cost overburden rate, total and below 60% of median income | Annual (survey) |
| `ilc_di03` | Median equivalised net income | Annual (survey) |
| `sts_cobp_a` | Building permits, dwellings in residential buildings | Annual |

One detail that matters: in 2026 Eurostat froze the old HICP table (`prc_hicp_midx`) at December 2025 and moved to
`prc_hicp_minr`. A project that kept the old code would silently stop at 2025.

In [2]:
raw = {f.stem: pd.read_csv(f, dtype=str) for f in sorted(RAW.glob("*.csv"))}
pd.DataFrame({"rows": {k: len(v) for k, v in raw.items()},
              "columns": {k: ", ".join(v.columns) for k, v in raw.items()}})

,rows,columns
ilc_di03,446,"freq, age, sex, statinfo, unit, geo, time_peri..."
ilc_lvho07a,489,"freq, unit, rskpovth, age, sex, geo, time_peri..."
prc_hicp_minr,5810,"freq, unit, coicop18, geo, time_period, obs_va..."
prc_hpi_q,1881,"freq, purchase, unit, geo, time_period, obs_va..."
sts_cobp_a,4140,"freq, indic_bt, cpa2_1, s_adj, unit, geo, time..."


## Step 2. Load the raw files and check them

Official data have their own traps: provisional (`p`) and estimated (`e`) values, series that start in different
years, and gaps. The checks below say what the model can rely on.

In [3]:
hpi = raw["prc_hpi_q"].query("unit == 'I15_Q'").copy()
hpi["quarter_start"] = pd.PeriodIndex(hpi["time_period"].str.replace("-", ""), freq="Q").start_time
hpi["value"] = hpi["obs_value"].astype(float)
hpi = hpi[hpi["quarter_start"] >= "2005-01-01"]

coverage = hpi[hpi["purchase"] == "TOTAL"].groupby("geo").agg(
    first=("quarter_start", "min"), last=("quarter_start", "max"), quarters=("value", "count"),
    provisional=("obs_flag", lambda f: (f == "p").sum()))
coverage.index = coverage.index.map(NAMES)
coverage

,first,last,quarters,provisional
geo,,,,
Germany,2005-01-01,2026-04-01,86,2
Spain,2005-10-01,2026-04-01,83,0
EU-27,2005-01-01,2026-04-01,86,0
France,2005-01-01,2026-04-01,86,1
Ireland,2005-01-01,2026-04-01,86,1
Italy,2010-01-01,2026-04-01,66,1
Netherlands,2005-01-01,2026-04-01,86,1
Portugal,2008-01-01,2026-04-01,74,0


In [4]:
flags = pd.concat({k: v["obs_flag"].fillna("(none)").value_counts() for k, v in raw.items()}, axis=1).fillna(0).astype(int)
flags.index.name = "status flag"
flags   # p = provisional, e = estimated, b = break in series, u = low reliability

,ilc_di03,ilc_lvho07a,prc_hicp_minr,prc_hpi_q,sts_cobp_a
status flag,,,,,
(none),404,450,5330,1864,3652
b,30,24,0,0,0
e,12,15,480,0,180
p,0,0,0,17,136
i,0,0,0,0,172


## Step 3. Model the indices in dbt

The SQL lives in [`models/`](../models). Staging views type and filter the raw tables; the marts form a small star
schema (`dim_country`, `dim_quarter`, `fct_house_prices`, `fct_consumer_prices`, `fct_living_conditions`,
`fct_building_permits`) and two marts answer the question:

* `mart_house_prices_real`: monthly HICP averaged to quarters (only complete quarters), then
  **real house prices = house price index ÷ all-items HICP × 100**, plus rents and year-on-year changes.
* `mart_affordability`: **price-to-income = annual average house price index ÷ median income indexed to 2015 × 100**.
  EU-SILC asks about the previous year's income, so the survey of year *t* is aligned to income year *t − 1* before
  it is compared with prices. That is why the latest price-to-income year is 2024, not 2025.

`dbt build` runs 38 nodes (models and tests: keys, relationships, accepted values and a check that every index
averages 100 in its base year) with no failures. The same steps in pandas:

In [5]:
cpi = raw["prc_hicp_minr"].query("unit == 'I15'").copy()
cpi["month"] = pd.to_datetime(cpi["time_period"].str.replace("M", "-") + "-01")
cpi["value"] = cpi["obs_value"].astype(float)
cpi = cpi[cpi["month"] >= "2005-01-01"]
cpi["quarter_start"] = cpi["month"].dt.to_period("Q").dt.start_time
cpi_q = cpi.groupby(["geo", "coicop18", "quarter_start"])["value"].agg(["mean", "count"])
cpi_q = cpi_q[cpi_q["count"] == 3]["mean"].unstack("coicop18").rename(columns={"TOTAL": "cpi", "CP041": "rent"})

q = hpi[hpi["purchase"] == "TOTAL"].set_index(["geo", "quarter_start"])["value"].rename("nominal").to_frame().join(cpi_q, how="inner")
q["real"] = q["nominal"] / q["cpi"] * 100
q = q.reset_index()
q["country"] = q["geo"].map(NAMES)
q["nominal_yoy"] = q.groupby("geo")["nominal"].pct_change(4)
q["real_yoy"] = q.groupby("geo")["real"].pct_change(4)
last = q["quarter_start"].max()
q[q["quarter_start"] == last].set_index("country")[["nominal", "real", "nominal_yoy", "real_yoy"]].sort_values("nominal", ascending=False) \
    .style.format({"nominal": "{:.1f}", "real": "{:.1f}", "nominal_yoy": "{:+.1%}", "real_yoy": "{:+.1%}"})

,nominal,real,nominal_yoy,real_yoy
country,,,,
Portugal,301.4,232.4,+16.5%,+12.9%
Netherlands,222.6,159.2,+4.3%,+1.5%
Ireland,205.9,163.6,+6.0%,+2.4%
Spain,200.6,152.8,+12.1%,+8.3%
EU-27,168.6,123.0,+4.7%,+1.5%
Germany,153.5,113.5,+0.6%,-2.0%
France,125.5,98.4,-0.8%,-3.1%
Italy,121.0,94.0,+4.0%,+0.9%


## Step 4. Answer the question

### House prices since 2015, against the peers

**Switch between real and nominal** with the buttons under the chart; **click a country in the legend** to highlight
it. Real prices remove consumer inflation, which matters: Portugal's index tripled in nominal terms and more than
doubled in real terms.

In [6]:
long = q[q["quarter_start"] >= "2015-01-01"].melt(id_vars=["country", "quarter_start"], value_vars=["nominal", "real"],
                                                  var_name="measure", value_name="index")
measure = alt.param(name="measure", value="real", bind=alt.binding_radio(options=["real", "nominal"], name="Prices: "))
country = alt.selection_point(fields=["country"], bind="legend", value=[{"country": "Portugal"}])
alt.Chart(long).transform_filter("datum.measure == measure").mark_line(strokeWidth=2).encode(
    x=alt.X("quarter_start:T", title=None),
    y=alt.Y("index:Q", title="House price index, 2015 = 100"),
    color=alt.Color("country:N", scale=PALETTE, title="Country (click)"),
    opacity=alt.condition(country, alt.value(1), alt.value(0.15)),
    strokeWidth=alt.condition(country, alt.value(3), alt.value(1.5)),
    tooltip=["country", alt.Tooltip("quarter_start:T", title="Quarter", format="%Y Q%q"), alt.Tooltip("index:Q", format=".1f")],
).add_params(measure, country).properties(width=560, height=280, title="House prices, 2015 = 100")

alt.Chart(...)

### Prices, rents and the cost of living in Portugal

Rents in the HICP cover all tenancies, old contracts included, so they rise far more slowly than asking rents for new
leases; still, the gap with house prices is large. Hover for the values.

In [7]:
pt = q[(q["geo"] == "PT") & (q["quarter_start"] >= "2015-01-01")].melt(
    id_vars="quarter_start", value_vars=["nominal", "rent", "cpi"], var_name="series", value_name="index")
pt["series"] = pt["series"].map({"nominal": "House prices", "rent": "Rents (HICP)", "cpi": "All consumer prices"})
hover = alt.selection_point(fields=["quarter_start"], nearest=True, on="pointerover", empty=False)
base = alt.Chart(pt).encode(x=alt.X("quarter_start:T", title=None))
lines = base.mark_line(strokeWidth=2).encode(
    y=alt.Y("index:Q", title="Index, 2015 = 100"),
    color=alt.Color("series:N", scale=alt.Scale(range=[ORANGE, BLUE, MUTED]), title=None))
points = base.mark_point(filled=True, size=60).encode(
    y="index:Q", color=alt.Color("series:N", legend=None),
    opacity=alt.condition(hover, alt.value(1), alt.value(0)),
    tooltip=[alt.Tooltip("quarter_start:T", format="%Y Q%q", title="Quarter"), "series", alt.Tooltip("index:Q", format=".1f")],
).add_params(hover)
rule = base.mark_rule(color=GREY).encode(opacity=alt.condition(hover, alt.value(1), alt.value(0))).transform_filter(hover)
(lines + points + rule).properties(width=560, height=260, title="Portugal: house prices, rents and consumer prices")

alt.LayerChart(...)

### Have prices outrun incomes?

**Click a country** on the left: the right chart shows its house price index against its median income, both
indexed to 2015. Where the orange line pulls away from the blue one, housing became less affordable.

In [8]:
ann = q.assign(year=q["quarter_start"].dt.year).groupby(["geo", "year"])["nominal"].agg(["mean", "count"])
ann = ann[ann["count"] == 4]["mean"].rename("hpi").reset_index()
inc = raw["ilc_di03"].query("statinfo == 'MED_EI' and unit == 'EUR' and age == 'TOTAL' and sex == 'T'").copy()
inc["year"] = inc["time_period"].astype(int) - 1                # survey year t reports income of year t - 1
inc["income"] = inc["obs_value"].astype(float)
aff = ann.merge(inc[["geo", "year", "income"]], on=["geo", "year"])
aff["income_2015"] = aff["income"] / aff.groupby("geo")["income"].transform(lambda s: s[aff.loc[s.index, "year"] == 2015].iloc[0]) * 100
aff["pti"] = aff["hpi"] / aff["income_2015"] * 100
aff["country"] = aff["geo"].map(NAMES)
pti_last = aff[aff["year"] == aff["year"].max()].sort_values("pti", ascending=False)

pick = alt.selection_point(name="pick", fields=["country"], on="click", clear="dblclick", value=[{"country": "Portugal"}])
bars = alt.Chart(pti_last).mark_bar(cornerRadius=3).encode(
    y=alt.Y("country:N", sort="-x", title=None),
    x=alt.X("pti:Q", title=f"Price-to-income index, {int(pti_last['year'].iloc[0])} (2015 = 100)"),
    color=alt.condition(pick, alt.value(ORANGE), alt.value(GREY)),
    tooltip=["country", alt.Tooltip("pti:Q", format=".1f", title="Price-to-income")],
).add_params(pick).properties(width=250, height=220, title="Click a country")
rule100 = alt.Chart(pd.DataFrame({"x": [100]})).mark_rule(strokeDash=[3, 3], color=MUTED).encode(x="x:Q")

two = aff[aff["year"] >= 2015].melt(id_vars=["country", "year"], value_vars=["hpi", "income_2015"], var_name="series", value_name="index")
two["series"] = two["series"].map({"hpi": "House prices", "income_2015": "Median income"})
chosen = "(pick.country && length(pick.country)) ? pick.country[0] : 'Portugal'"
lines = alt.Chart(two).transform_filter(f"datum.country == ({chosen})").mark_line(point=True, strokeWidth=2).encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("index:Q", title="Index, 2015 = 100", scale=alt.Scale(zero=False)),
    color=alt.Color("series:N", scale=alt.Scale(range=[ORANGE, BLUE]), title=None),
    tooltip=["country", "series", "year", alt.Tooltip("index:Q", format=".1f")],
).properties(width=380, height=220, title=alt.Title(alt.expr(f"'House prices and median income · ' + ({chosen})")))
alt.hconcat(bars + rule100, lines)

alt.HConcatChart(...)

### Has building caught up, and who is overburdened?

Permits for new homes in Portugal recovered from the 2014 low but remain well below 2005. The overburden rate is the
share of people spending more than 40% of disposable income on housing. **Drag across the permits chart** to pick
years; the right chart shows the overburden rates for the same years.

In [9]:
per = raw["sts_cobp_a"].query("indic_bt == 'BPRM_DW' and cpa2_1 == 'CPA_F41001_X_410014' and unit == 'THS' and geo == 'PT'").copy()
per["year"] = per["time_period"].astype(int)
per["dwellings"] = per["obs_value"].astype(float) * 1000
ob = raw["ilc_lvho07a"].query("unit == 'PC' and age == 'TOTAL' and sex == 'T' and geo in ['PT', 'EU27_2020'] and rskpovth in ['TOTAL', 'B_60']").copy()
ob["year"] = ob["time_period"].astype(int)
ob["rate"] = ob["obs_value"].astype(float) / 100
ob["group"] = ob["geo"].map(NAMES) + " · " + ob["rskpovth"].map({"TOTAL": "everyone", "B_60": "low income"})

years = alt.selection_interval(encodings=["x"], value={"x": [2015, 2025]})
permits = alt.Chart(per).mark_bar(color=BLUE, size=11).encode(
    x=alt.X("year:Q", title=None, axis=alt.Axis(format="d"), scale=alt.Scale(domain=[2004.5, 2025.5])),
    y=alt.Y("dwellings:Q", title="Dwellings permitted", axis=alt.Axis(format="~s")),
    color=alt.condition(years, alt.value(BLUE), alt.value(GREY)),
    tooltip=["year", alt.Tooltip("dwellings:Q", format=",.0f")],
).add_params(years).properties(width=330, height=220, title="Portugal: new homes permitted (drag to pick years)")
burden = alt.Chart(ob).transform_filter(years).mark_line(point=True).encode(
    x=alt.X("year:Q", title="Survey year", axis=alt.Axis(format="d"), scale=alt.Scale(zero=False)),
    y=alt.Y("rate:Q", title="Overburdened by housing costs", axis=alt.Axis(format="%")),
    color=alt.Color("group:N", title=None, legend=alt.Legend(orient="bottom", columns=2),
                    scale=alt.Scale(domain=["Portugal · everyone", "Portugal · low income", "EU-27 · everyone", "EU-27 · low income"],
                                    range=[ORANGE, "#f2b08f", MUTED, GREY])),
    strokeDash=alt.StrokeDash("group:N", legend=None),
    tooltip=["group", "year", alt.Tooltip("rate:Q", format=".1%")],
).properties(width=330, height=220, title="Overburden rate in the picked years")
alt.hconcat(permits, burden)

alt.HConcatChart(...)

**Answer.** House prices in Portugal have tripled since 2015 (2.3 times after inflation), the steepest rise of the
eight; from 2015 to 2024 they rose 124% while median income rose 65%, so the price-to-income index (136) is the
highest of the group. Building recovered but is still 45% below 2005. The average household is not where the
pressure shows: only 6.3% of people are overburdened, below the EU's 7.7%, because most own their home outright;
among low-income households it is about one in four.

## Step 5. Reconcile pandas with the dbt marts and the reference numbers

The README publishes reference numbers so that the Power BI and Qlik versions can be checked after their first load.
Here pandas, the dbt marts and those numbers must all agree.

In [10]:
db = duckdb.connect("../data/housing.duckdb", read_only=True)
m_real = db.sql("select geo, quarter_start, nominal_hpi, real_hpi, nominal_yoy from mart_house_prices_real").df()
m_aff = db.sql("select geo, year, price_to_income_index from mart_affordability where price_to_income_index is not null").df()
db.close()

ptq = q[(q["geo"] == "PT") & (q["quarter_start"] == "2026-04-01")].iloc[0]
mq = m_real[(m_real["geo"] == "PT") & (m_real["quarter_start"] == pd.Timestamp("2026-04-01"))].iloc[0]
pti_pt = aff[(aff["geo"] == "PT") & (aff["year"] == 2024)]["pti"].iloc[0]
m_pti = m_aff[(m_aff["geo"] == "PT") & (m_aff["year"] == 2024)]["price_to_income_index"].iloc[0]
merged = q.merge(m_real, on=["geo", "quarter_start"])
recon = pd.DataFrame([
    ["PT house price index, 2026 Q2 (nominal)", ptq["nominal"], mq["nominal_hpi"], 301.4],
    ["PT house price index, 2026 Q2 (real)", ptq["real"], mq["real_hpi"], 232.4],
    ["PT year-on-year, 2026 Q2 (%)", ptq["nominal_yoy"] * 100, mq["nominal_yoy"] * 100, 16.5],
    ["PT price-to-income index, 2024", pti_pt, m_pti, 136.3],
    ["PT dwellings permitted, 2025", per.loc[per["year"] == 2025, "dwellings"].iloc[0], None, 40_900],
    ["Largest real-index gap, any country and quarter", (merged["real"] - merged["real_hpi"]).abs().max(), 0, None],
], columns=["measure", "pandas", "dbt", "README"])
recon["match"] = [np.isclose(r.pandas, r.dbt if pd.notna(r.dbt) else r.pandas, atol=1e-6)
                  and (pd.isna(r.README) or abs(r.pandas - r.README) < 0.05 + 1e-9 * abs(r.README)) for r in recon.itertuples()]
assert recon["match"].all(), recon
recon

,measure,pandas,dbt,README,match
0,"PT house price index, 2026 Q2 (nominal)",301.4,301.4,301.4,True
1,"PT house price index, 2026 Q2 (real)",232.4,232.4,232.4,True
2,"PT year-on-year, 2026 Q2 (%)",16.5,16.5,16.5,True
3,"PT price-to-income index, 2024",136.3,136.3,136.3,True
4,"PT dwellings permitted, 2025","40,900.0",NaN,"40,900.0",True
5,"Largest real-index gap, any country and quarter",0.0,0.0,NaN,True


pandas and dbt agree on every quarter and country, and both match the published reference numbers.

## Step 6. Publish

* **Web dashboard** ([`web/`](../web), live at [guilhermerigonato.com/pt-housing-analytics](https://guilhermerigonato.com/pt-housing-analytics/)):
  [`scripts/export.py`](../scripts/export.py) writes JSON from the marts; clicking a chart pins a quarter or a year
  across the panels, and a country bar changes the comparison.
* **Power BI** ([`powerbi/`](../powerbi)) and **Qlik Sense** ([`qlik/`](../qlik)) load the Parquet files in
  [`exports/`](../exports), so neither needs DuckDB.

## Which library for interactive charts

| Library | Linked charts | Output | Best for |
|---|---|---|---|
| **Altair** (Vega-Lite), used here | Built in: selections and widgets shared across charts | Static HTML, no server | Reports that must open anywhere |
| **Plotly** | Per chart; linking across charts needs Dash | HTML; a Dash app for linking | Rich single charts, maps |
| **Bokeh** | Linked brushing through a shared data source | HTML, or a Bokeh server | Large scatter plots, custom tools |
| **HoloViews + Panel** | `link_selections` in one call | Notebook or Panel app | Turning a notebook into a small app |
| **Dash / Streamlit** | Any, through Python callbacks | Needs a server | Internal tools with live data |